# 05 信号研究とベースライン(共通信号スキーマ)

`irp.signals` は異なるファミリ(trend/value/quality/carry/risk/macro)を **共通の 比較可能な形** に揃える:各信号は横断標準化された **score** パネル・**category**・**direction** を持つ :class:`Signal`。標準化により合成は重み付き平均で済む。ここでは複数のベースライン信号を作り、合成し、ロング/ショート weight に落とし、**信号は使う前に lag** して同足の先読みを避けることを示す。

In [ ]:
import numpy as np
import pandas as pd

# Deterministic synthetic price panel (no network needed; the same APIs work
# on real data via irp.data.get_prices / price_panel). Five assets with
# different drift and volatility so feature/signal behavior is predictable.
rng = np.random.default_rng(7)
idx = pd.bdate_range('2018-01-01', periods=750)
drift = {'AAA': 0.0002, 'BBB': 0.0004, 'CCC': 0.0006, 'DDD': 0.0008, 'EEE': 0.0010}
vol = {'AAA': 0.008, 'BBB': 0.010, 'CCC': 0.012, 'DDD': 0.018, 'EEE': 0.025}
steps = {a: rng.normal(drift[a], vol[a], len(idx)) for a in drift}
prices = pd.DataFrame({a: 100 * np.exp(np.cumsum(s)) for a, s in steps.items()}, index=idx)
prices.tail(3)

## ベースライン信号(momentum / trend / low-vol / mean-reversion)

In [ ]:
from irp import signals as S

mom = S.momentum_signal(prices, lookback=252, skip=21)
trend = S.trend_following_signal(prices, window=200)
lowvol = S.low_volatility_signal(prices, window=63)
rev = S.mean_reversion_signal(prices, window=21)
for sig in (mom, trend, lowvol, rev):
    print(f'{sig.name:16s} category={sig.category.value:6s} direction={sig.direction:+d}')
print()
print('momentum oriented score (last date, higher = prefer long):')
display(mom.oriented.tail(1).T)

標準化済みなので各日の score は横断的に平均≈0。だから複数信号を **平均** すれば合成になる。

In [ ]:
comp = S.combine([mom, trend, lowvol], weights=[1.0, 1.0, 0.5], name='blend')
print('composite components:', comp.meta['components'], 'weights:', comp.meta['weights'])
print('row mean of a standardized signal (~0):', round(float(mom.score.tail(1).mean(axis=1).iloc[0]), 6))
display(comp.score.tail(1).T)

## 信号 → ロング/ショート weight(dollar-neutral、研究用の素朴版)

上位/下位 quantile を等加重で売買(net 0, gross 1+1)。これは信号の spread を見るための **研究用ユーティリティ** で、コスト・walk-forward を含む本格的なバックテスト(Phase 2b)ではない。

In [ ]:
w = S.long_short_quantile(comp.score, quantile=0.4)
last = w.dropna().tail(1)
print('weights (last date): net =', round(float(last.sum(axis=1).iloc[0]), 6),
      '| gross =', round(float(last.abs().sum(axis=1).iloc[0]), 6))
display(last.T)

## 先読み回避:信号は **lag** してから次足リターンに当てる

`Signal.lag(1)` で date-`t` の判断が date-`t-1` の情報だけを使うようにする。下は **研究用の診断**(本番バックテストではない):lag した合成信号の ロング/ショートを翌足リターンに当てた spread。

In [ ]:
from irp.labels import forward_return

lagged = comp.lag(1)                      # decide on yesterday's info
w = S.long_short_quantile(lagged.score, quantile=0.4)
fwd = forward_return(prices, horizon=1)   # next-bar return (label)
pnl = (w * fwd).sum(axis=1).dropna()      # daily long-short spread
ann = 252
sharpe = float(pnl.mean() / pnl.std() * np.sqrt(ann)) if pnl.std() else float('nan')
print(f'synthetic long-short spread — days: {len(pnl)}, '
      f'mean/day: {pnl.mean():+.5f}, naive ann.Sharpe: {sharpe:+.2f}')
print('NOTE: synthetic data + no costs/turnover/tax — illustrative only, not a result.')

## レジストリ:実装済み信号と、データ待ちの予定ファミリ

In [ ]:
print('implemented:', S.available())
from irp.signals import PLANNED
print('planned (need fundamentals/term-structure — Phase 2):', sorted(PLANNED))
try:
    S.get_signal('value')(prices)
except NotImplementedError as e:
    print('value ->', str(e).split('.')[0])

信号は全て同じスキーマなので、ベースラインも将来の ML/時系列モデルも **同じ土俵** で比較できる(プラットフォームの方針:複雑なモデルは必ず単純ベースラインと比較)。次は Phase 2b の walk-forward バックテスト(コスト/スリッページ/税/ターンオーバー/ベンチ比較)。